# Delta Lake — ACID, Time Travel, Z-ordering, OPTIMIZE

**Mental model:** Delta Lake is Parquet plus a transaction log. The table data sits in files, but the `_delta_log/` directory records every versioned change, which is what enables ACID guarantees, optimistic concurrency control, schema evolution, and time travel.

This notebook demonstrates Delta Lake internals from a Databricks SQL Warehouse using `requests` rather than `databricks-sdk`, grounded in a Citi-style telemetry and alerting narrative with monitored API endpoints, metrics, and alerts.


In [ ]:

import json
import time
from datetime import datetime, timezone
from pprint import pprint

import requests

HOST = "https://dbc-9f35a83d-b4e7.cloud.databricks.com"
TOKEN = ""
WAREHOUSE_ID = "b6657f31d1e7a179"

if not TOKEN:
    print("WARNING: Databricks TOKEN is blank. Set a personal access token before executing live SQL.")

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

SQL_ENDPOINT = f"{HOST}/api/2.0/sql/statements"

def execute_sql(sql: str, *, wait_timeout_s: int = 120, verbose: bool = False):
    payload = {
        "statement": sql,
        "warehouse_id": WAREHOUSE_ID,
        "wait_timeout": "0s",
        "disposition": "INLINE",
        "format": "JSON_ARRAY",
    }
    response = requests.post(SQL_ENDPOINT, headers=HEADERS, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    statement_id = data["statement_id"]
    status = data.get("status", {}).get("state", "PENDING")

    started = time.time()
    while status in {"PENDING", "RUNNING"}:
        if time.time() - started > wait_timeout_s:
            raise TimeoutError(f"Statement {statement_id} timed out after {wait_timeout_s}s")
        time.sleep(2)
        poll = requests.get(f"{SQL_ENDPOINT}/{statement_id}", headers=HEADERS, timeout=30)
        poll.raise_for_status()
        data = poll.json()
        status = data.get("status", {}).get("state", status)
        if verbose:
            print(f"Polling {statement_id}: {status}")

    if status != "SUCCEEDED":
        raise RuntimeError(json.dumps(data, indent=2))

    result = data.get("result", {})
    manifest = result.get("manifest", {})
    schema = manifest.get("schema", {})
    columns = [c.get("name") for c in schema.get("columns", [])]
    rows = result.get("data_array", [])
    return {
        "statement_id": statement_id,
        "status": status,
        "columns": columns,
        "rows": rows,
        "raw": data,
    }


def show_result(title: str, result: dict, limit: int = 20):
    print(f"\n=== {title} ===")
    if result["columns"]:
        print("Columns:", result["columns"])
    for row in result["rows"][:limit]:
        print(row)
    if len(result["rows"]) > limit:
        print(f"... showing first {limit} of {len(result['rows'])} rows")

print("Requests-based Databricks SQL executor ready")
print(f"HOST={HOST}")
print(f"WAREHOUSE_ID={WAREHOUSE_ID}")


In [ ]:

create_schema_sql = "CREATE SCHEMA IF NOT EXISTS citi_delta"
execute_sql(create_schema_sql)

drop_table_sql = "DROP TABLE IF EXISTS citi_delta.alerts"
execute_sql(drop_table_sql)

create_table_sql = """
CREATE TABLE citi_delta.alerts (
    alert_id INT,
    endpoint_id INT,
    severity STRING,
    message STRING,
    created_at TIMESTAMP
)
USING DELTA
"""
execute_sql(create_table_sql)

seed_sql = """
INSERT INTO citi_delta.alerts VALUES
(1, 1001, 'CRITICAL', 'Latency spike detected on payments-auth', TIMESTAMP '2026-03-30 09:00:00'),
(2, 1002, 'HIGH',     'Error rate breach on customer-profile', TIMESTAMP '2026-03-30 09:02:00'),
(3, 1003, 'MEDIUM',   'Throughput dip on card-ledger', TIMESTAMP '2026-03-30 09:04:00'),
(4, 1004, 'LOW',      'Intermittent timeout on fx-pricing', TIMESTAMP '2026-03-30 09:06:00'),
(5, 1005, 'CRITICAL', 'Sustained 5xx burst on transfers-api', TIMESTAMP '2026-03-30 09:08:00'),
(6, 1006, 'HIGH',     'P99 latency degradation on statements', TIMESTAMP '2026-03-30 09:10:00'),
(7, 1007, 'MEDIUM',   'Queue lag warning on notifications', TIMESTAMP '2026-03-30 09:12:00'),
(8, 1008, 'LOW',      'Minor retry increase on balances-api', TIMESTAMP '2026-03-30 09:14:00'),
(9, 1009, 'CRITICAL', 'Auth dependency failure on mobile-login', TIMESTAMP '2026-03-30 09:16:00'),
(10, 1010, 'HIGH',    'Downstream saturation on fraud-check', TIMESTAMP '2026-03-30 09:18:00'),
(11, 1011, 'MEDIUM',  'CPU pressure on quote-enricher', TIMESTAMP '2026-03-30 09:20:00'),
(12, 1012, 'LOW',     'Elevated garbage collection on customer-docs', TIMESTAMP '2026-03-30 09:22:00'),
(13, 1013, 'CRITICAL','Regional outage signal on wires-api', TIMESTAMP '2026-03-30 09:24:00'),
(14, 1014, 'HIGH',    'Thread pool exhaustion on limits-engine', TIMESTAMP '2026-03-30 09:26:00'),
(15, 1015, 'MEDIUM',  'Cache miss surge on beneficiary-service', TIMESTAMP '2026-03-30 09:28:00'),
(16, 1016, 'LOW',     'Minor packet loss on card-activation', TIMESTAMP '2026-03-30 09:30:00'),
(17, 1017, 'CRITICAL','Payment switch unavailable on card-swipe', TIMESTAMP '2026-03-30 09:32:00'),
(18, 1018, 'HIGH',    'Memory growth on aml-screening', TIMESTAMP '2026-03-30 09:34:00'),
(19, 1019, 'MEDIUM',  'Replica lag on transaction-history', TIMESTAMP '2026-03-30 09:36:00'),
(20, 1020, 'LOW',     'Low-volume anomaly on rewards-sync', TIMESTAMP '2026-03-30 09:38:00')
"""
execute_sql(seed_sql, wait_timeout_s=180)

count_result = execute_sql("SELECT COUNT(*) AS row_count FROM citi_delta.alerts")
show_result("Initial Delta table count", count_result)
print("Delta table created")


In [ ]:

insert_txn_1 = """
INSERT INTO citi_delta.alerts VALUES
(21, 1021, 'CRITICAL', 'API gateway saturation on retail-payments', current_timestamp())
"""
insert_txn_2 = """
INSERT INTO citi_delta.alerts VALUES
(22, 1022, 'HIGH', 'Recovery warning on treasury-positions', current_timestamp())
"""

execute_sql(insert_txn_1)
execute_sql(insert_txn_2)

acid_result = execute_sql("SELECT alert_id, endpoint_id, severity, message FROM citi_delta.alerts WHERE alert_id IN (21, 22) ORDER BY alert_id")
show_result("Sequential writes committed", acid_result)

print(
    "Delta uses optimistic concurrency control. Each writer validates against the latest transaction log version before commit. "
    "If two writers attempt incompatible changes, Delta rejects the losing transaction instead of corrupting the table. "
    "These two sequential INSERTs both succeed because they commit cleanly as separate log versions."
)


In [ ]:

more_rows_sql = """
INSERT INTO citi_delta.alerts VALUES
(23, 1023, 'MEDIUM', 'Latency recovered but still elevated on customer-360', current_timestamp()),
(24, 1024, 'LOW',    'Background reconciliation delay on settlements', current_timestamp()),
(25, 1025, 'HIGH',   'Error budget burn on api-orchestrator', current_timestamp()),
(26, 1026, 'CRITICAL','Payment timeout cluster on mobile-wallet', current_timestamp()),
(27, 1027, 'LOW',    'Retry pattern normalized on card-controls', current_timestamp())
"""
execute_sql(more_rows_sql)

history_result = execute_sql("DESCRIBE HISTORY citi_delta.alerts")
show_result("Delta history", history_result, limit=10)

original_version_result = execute_sql(
    "SELECT alert_id, endpoint_id, severity, message FROM citi_delta.alerts VERSION AS OF 0 ORDER BY alert_id"
)
show_result("VERSION AS OF 0 (original state)", original_version_result, limit=25)

history_rows = history_result["rows"]
timestamp_as_of = None
for row in history_rows:
    if row[0] == 0:
        timestamp_as_of = row[1]
        break

if timestamp_as_of is None:
    raise RuntimeError("Could not find version 0 timestamp from DESCRIBE HISTORY")

time_based_result = execute_sql(
    f"SELECT alert_id, endpoint_id, severity, message FROM citi_delta.alerts TIMESTAMP AS OF '{timestamp_as_of}' ORDER BY alert_id"
)
show_result("TIMESTAMP AS OF original version timestamp", time_based_result, limit=25)


In [ ]:

execute_sql("ALTER TABLE citi_delta.alerts ADD COLUMNS (response_time_ms DOUBLE)")

schema_insert_sql = """
INSERT INTO citi_delta.alerts (alert_id, endpoint_id, severity, message, created_at, response_time_ms) VALUES
(28, 1028, 'HIGH', 'Response latency above SLO on trade-capture', current_timestamp(), 842.5),
(29, 1029, 'MEDIUM', 'Transient slowness on custody-events', current_timestamp(), 410.2),
(30, 1030, 'LOW', 'Healthy but monitored variance on batch-reporter', current_timestamp(), 180.0)
"""
execute_sql(schema_insert_sql)

schema_result = execute_sql(
    "SELECT alert_id, endpoint_id, severity, response_time_ms FROM citi_delta.alerts WHERE alert_id BETWEEN 27 AND 30 ORDER BY alert_id"
)
show_result("Schema evolution result", schema_result)

print(
    "Delta schema evolution is metadata-safe. Older rows remain readable with NULL for the new column, "
    "while newer writes can populate response_time_ms without rewriting the whole table definition manually."
)


In [ ]:

filtered_before = execute_sql(
    "SELECT severity, endpoint_id, alert_id, message FROM citi_delta.alerts WHERE severity = 'CRITICAL' AND endpoint_id BETWEEN 1000 AND 1030 ORDER BY endpoint_id"
)
show_result("Filtered query before OPTIMIZE", filtered_before, limit=20)

optimize_result = execute_sql("OPTIMIZE citi_delta.alerts ZORDER BY (severity, endpoint_id)", wait_timeout_s=300)
show_result("OPTIMIZE ZORDER output", optimize_result, limit=20)

filtered_after = execute_sql(
    "SELECT severity, endpoint_id, alert_id, message FROM citi_delta.alerts WHERE severity = 'CRITICAL' AND endpoint_id BETWEEN 1000 AND 1030 ORDER BY endpoint_id"
)
show_result("Filtered query after OPTIMIZE", filtered_after, limit=20)

print(
    "Z-ordering co-locates related values across files so Databricks can skip more data during filtered reads. "
    "For Citi-style alert analysis, clustering by severity and endpoint_id improves pruning for hot operational queries."
)


## What Just Happened

- **Delta Lake = Parquet + transaction log.** The `_delta_log/` directory is the entire ACID story.
- **Concurrent safety comes from optimistic concurrency.** Writers commit new versions, and conflicting commits are rejected rather than silently merged into corruption.
- **Time travel is just reading an older log version.** `VERSION AS OF` and `TIMESTAMP AS OF` both replay a prior snapshot.
- **Schema evolution is controlled metadata change.** New columns can be added without breaking old reads.
- **OPTIMIZE + ZORDER improve file layout.** They do not change query semantics; they change how efficiently the engine can prune files and scan data.

Citi uses Delta on Databricks as the production lakehouse format because it adds operational reliability, auditability, and recovery semantics to raw Parquet storage.
